# 🏎️ McQueen StyleTTS2 — Full Fresh Training (250 Epochs)
**No Google Drive needed.** Just upload `mcqueen_audio.zip` when prompted.

Steps: Setup → Upload WAVs → Auto-transcribe → Train Stage 1 → Train Stage 2 (250 epochs) → Download

**Requirements:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1 — GPU CHECK
# ═══════════════════════════════════════════════════════════════════
import torch
assert torch.cuda.is_available(), '❌ NO GPU — Go to Runtime > Change runtime type > T4 GPU'
print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — INSTALL EVERYTHING
# ═══════════════════════════════════════════════════════════════════
!apt-get install -qq espeak-ng ffmpeg
!pip install -q \
    SoundFile phonemizer munch einops tqdm librosa \
    transformers accelerate openai-whisper \
    torch==2.1.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
print('✅ Dependencies installed')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — CLONE STYLETTS2
# ═══════════════════════════════════════════════════════════════════
import os
!git clone -q https://github.com/yl4579/StyleTTS2 /content/StyleTTS2
os.chdir('/content/StyleTTS2')
!pip install -q -r requirements.txt
print('✅ StyleTTS2 cloned')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4 — DOWNLOAD PRETRAINED UTILITY MODELS
# ═══════════════════════════════════════════════════════════════════
import os

os.makedirs('Utils/ASR', exist_ok=True)
os.makedirs('Utils/JDC', exist_ok=True)
os.makedirs('Utils/PLBERT', exist_ok=True)

# ASR (speech encoder)
if not os.path.exists('Utils/ASR/epoch_00080.pth'):
    !wget -q --show-progress -O Utils/ASR/epoch_00080.pth \
      'https://huggingface.co/yl4579/StyleTTS2-LibriTTS/resolve/main/Models/LibriTTS/epoch_00080.pth'

# JDC F0 pitch model
if not os.path.exists('Utils/JDC/bst.t7'):
    !wget -q --show-progress -O Utils/JDC/bst.t7 \
      'https://github.com/nickoala/jdc/raw/master/bst.t7'

# PLBERT (language model)
if not os.path.exists('Utils/PLBERT/config.json'):
    !git clone -q https://huggingface.co/yl4579/StyleTTS2-LibriTTS /tmp/s2pretrained
    !cp -r /tmp/s2pretrained/Utils/PLBERT Utils/

print('✅ All utility models ready')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5 — UPLOAD YOUR AUDIO ZIP
# Upload: mcqueen_audio.zip (the 12MB zip of your WAV files)
# ═══════════════════════════════════════════════════════════════════
from google.colab import files
import zipfile, os, glob

print('📁 Upload mcqueen_audio.zip when the dialog appears...')
uploaded = files.upload()  # <-- upload mcqueen_audio.zip here

zip_name = list(uploaded.keys())[0]
os.makedirs('/content/mcqueen_wavs', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/mcqueen_wavs')

wavs = glob.glob('/content/mcqueen_wavs/**/*.wav', recursive=True) + \
       glob.glob('/content/mcqueen_wavs/*.wav')
print(f'✅ Extracted {len(wavs)} WAV files')
for w in wavs[:5]:
    print(' ', w)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6 — AUTO-TRANSCRIBE WITH WHISPER
# ═══════════════════════════════════════════════════════════════════
import whisper, json, os, glob, torchaudio

model_whisper = whisper.load_model('base')
wavs = glob.glob('/content/mcqueen_wavs/**/*.wav', recursive=True) + \
       glob.glob('/content/mcqueen_wavs/*.wav')

data = []
bad = []
for wav_path in sorted(wavs):
    result = model_whisper.transcribe(wav_path, language='en')
    text = result['text'].strip()
    if not text:
        bad.append(wav_path)
        continue
    # Get duration
    info = torchaudio.info(wav_path)
    dur = info.num_frames / info.sample_rate
    data.append({'audio': wav_path, 'text': text, 'duration': round(dur, 2)})
    print(f'[{len(data):02d}] {os.path.basename(wav_path)}: "{text[:60]}"')

print(f'\n✅ Transcribed {len(data)} files, skipped {len(bad)} empty')

# Save metadata
with open('/content/mcqueen_meta.json', 'w') as f:
    json.dump(data, f, indent=2)
print('Saved /content/mcqueen_meta.json')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 7 — BUILD FILELIST (StyleTTS2 format)
# ═══════════════════════════════════════════════════════════════════
import json, os, random, shutil

with open('/content/mcqueen_meta.json') as f:
    data = json.load(f)

random.seed(42)
random.shuffle(data)
split = max(1, int(len(data) * 0.1))  # 10% validation
val_data = data[:split]
train_data = data[split:]

os.makedirs('/content/StyleTTS2/Data/McQueen', exist_ok=True)

def write_filelist(items, path):
    with open(path, 'w') as f:
        for item in items:
            f.write(f"{item['audio']}|{item['text']}\n")

write_filelist(train_data, '/content/StyleTTS2/Data/McQueen/train_list.txt')
write_filelist(val_data,   '/content/StyleTTS2/Data/McQueen/val_list.txt')

print(f'✅ Train: {len(train_data)} files, Val: {len(val_data)} files')
!head -3 /content/StyleTTS2/Data/McQueen/train_list.txt

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 8 — WRITE CONFIG FILES
# ═══════════════════════════════════════════════════════════════════
import os, yaml

os.makedirs('/content/StyleTTS2/Models/McQueen', exist_ok=True)

# ── Stage 1 config ────────────────────────────────────────────────
cfg1 = '''log_dir: 'Models/McQueen/'
save_freq: 5
log_interval: 10
device: 'cuda'
epochs_1st: 50
batch_size: 4
max_len: 80
pretrained_model: ''
second_stage_load_pretrained: true
load_only_params: true

F0_path: 'Utils/JDC/bst.t7'
ASR_config: 'Utils/ASR/config.yml'
ASR_path: 'Utils/ASR/epoch_00080.pth'
PLBERT_dir: 'Utils/PLBERT/'

data_params:
  train_data: 'Data/McQueen/train_list.txt'
  val_data: 'Data/McQueen/val_list.txt'
  root_path: ''
  OOD_data: 'Data/OOD_texts.txt'
  min_length: 50

preprocess_params:
  sr: 24000
  spect_params:
    n_fft: 2048
    win_length: 1200
    hop_length: 300

model_params:
  multispeaker: false
  dim_in: 64
  hidden_dim: 512
  max_conv_dim: 512
  n_layer: 3
  n_mels: 80
  n_token: 178
  max_dur: 50
  style_dim: 128
  dropout: 0.2
  asr_params:
    input_dim: 512
    n_token: 178
    token_embedding_dim: 512
    hidden_dim: 512
    n_layer: 3
    dropout: 0.1
    token_embedding_dim: 512
  decoder:
    type: 'istftnet'
    resblock_kernel_sizes: [3,7,11]
    upsample_rates: [10, 6]
    upsample_initial_channel: 512
    resblock_dilation_sizes: [[1,3,5], [1,3,5], [1,3,5]]
    upsample_kernel_sizes: [20, 12]
    gen_istft_n_fft: 20
    gen_istft_hop_size: 5
  slm:
    model: 'microsoft/wavlm-base-plus'
    sr: 16000
    hidden: 768
    nlayers: 13
    initial_channel: 64

loss_params:
  lambda_mel: 5.
  lambda_gen: 1.
  lambda_slm: 1.
  lambda_mono: 1.
  lambda_s2s: 1.

optimizer_params:
  lr: 0.0001

slmadv_params:
  min_len: 400
  max_len: 500
  batch_percentage: 0.5
  iter: 10
  thresh: 5
  scale: 0.01
  sig: 1.5
'''

# ── Stage 2 config ────────────────────────────────────────────────
cfg2 = '''log_dir: 'Models/McQueen/'
save_freq: 10
log_interval: 10
device: 'cuda'
max_epoch: 250
batch_size: 4
max_len: 80
pretrained_model: ''
second_stage_load_pretrained: true
load_only_params: true

F0_path: 'Utils/JDC/bst.t7'
ASR_config: 'Utils/ASR/config.yml'
ASR_path: 'Utils/ASR/epoch_00080.pth'
PLBERT_dir: 'Utils/PLBERT/'

data_params:
  train_data: 'Data/McQueen/train_list.txt'
  val_data: 'Data/McQueen/val_list.txt'
  root_path: ''
  OOD_data: 'Data/OOD_texts.txt'
  min_length: 50

preprocess_params:
  sr: 24000
  spect_params:
    n_fft: 2048
    win_length: 1200
    hop_length: 300

model_params:
  multispeaker: false
  dim_in: 64
  hidden_dim: 512
  max_conv_dim: 512
  n_layer: 3
  n_mels: 80
  n_token: 178
  max_dur: 50
  style_dim: 128
  dropout: 0.2
  asr_params:
    input_dim: 512
    n_token: 178
    token_embedding_dim: 512
    hidden_dim: 512
    n_layer: 3
    dropout: 0.1
    token_embedding_dim: 512
  decoder:
    type: 'istftnet'
    resblock_kernel_sizes: [3,7,11]
    upsample_rates: [10, 6]
    upsample_initial_channel: 512
    resblock_dilation_sizes: [[1,3,5], [1,3,5], [1,3,5]]
    upsample_kernel_sizes: [20, 12]
    gen_istft_n_fft: 20
    gen_istft_hop_size: 5
  slm:
    model: 'microsoft/wavlm-base-plus'
    sr: 16000
    hidden: 768
    nlayers: 13
    initial_channel: 64
  diffusion:
    embed_dim: 64
    embedding_mask_proba: 0.1
    unet:
      dim: 512
      resnet_groups: 8
      num_channels: 256
    noise_scheduler:
      type: 'vp'

loss_params:
  lambda_mel: 5.
  lambda_gen: 1.
  lambda_slm: 1.
  lambda_diff: 1.

optimizer_params:
  lr: 0.0001
  bert_lr: 0.00001
  ft_lr: 0.00001

slmadv_params:
  min_len: 400
  max_len: 500
  batch_percentage: 0.5
  iter: 10
  thresh: 5
  scale: 0.01
  sig: 1.5
'''

with open('/content/StyleTTS2/Models/McQueen/config_stage1.yml', 'w') as f:
    f.write(cfg1)
with open('/content/StyleTTS2/Models/McQueen/config_stage2.yml', 'w') as f:
    f.write(cfg2)

print('✅ Config files written')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 9 — STAGE 1 TRAINING (~45 mins)
# Trains the base acoustic model
# ═══════════════════════════════════════════════════════════════════
import os
os.chdir('/content/StyleTTS2')
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

!python train_first.py -p Models/McQueen/config_stage1.yml

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 10 — STAGE 2 TRAINING (250 epochs, ~2-3 hrs)
# Fine-tunes with diffusion for voice quality
# ═══════════════════════════════════════════════════════════════════
import os
os.chdir('/content/StyleTTS2')

!python train_second.py -p Models/McQueen/config_stage2.yml

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 11 — PACKAGE & DOWNLOAD
# Run this after training completes (or any time to grab current progress)
# ═══════════════════════════════════════════════════════════════════
import glob, re, torch, io, shutil, os
from google.colab import files

# Find latest stage 2 checkpoint
ckpts = glob.glob('/content/StyleTTS2/Models/McQueen/epoch_2nd_*.pth')
if not ckpts:
    print('No stage 2 checkpoint yet — check if training finished')
else:
    latest = sorted(ckpts, key=lambda x: int(re.search(r'epoch_2nd_(\d+)', x).group(1)))[-1]
    ep = int(re.search(r'epoch_2nd_(\d+)', latest).group(1))
    print(f'Packaging epoch {ep}...')

    # Prune to ~750MB (remove discriminators & optimizer states)
    full = torch.load(latest, map_location='cpu', weights_only=False)
    net = full.get('net', full)
    keep = ['bert', 'bert_encoder', 'predictor', 'text_encoder', 'decoder', 'diffusion']
    pruned = {k: v for k, v in net.items() if any(g in k for g in keep)}
    out_pth = f'/content/mcqueen_ep{ep}_pruned.pth'
    torch.save({'net': pruned}, out_pth)
    shutil.copy('/content/StyleTTS2/Models/McQueen/config_stage2.yml', '/content/config.yml')

    zip_name = f'/content/mcqueen_fresh_ep{ep}.zip'
    os.system(f'zip {zip_name} {out_pth} /content/config.yml')
    size = os.path.getsize(zip_name) / 1e6
    print(f'✅ {zip_name} ({size:.0f} MB) — downloading...')
    files.download(zip_name)